# MovieLens Recommender — Stage 3: Two-Tower Retrieval (PyTorch)

Scores so far:
- Popularity baseline: **Recall@10 0.0700 / NDCG@10 0.0632**
- Matrix factorization: **Recall@10 0.0825 / NDCG@10 0.0743** (+18%)

Now the modern architecture: **two towers**.
- A **user tower** maps a user -> an embedding.
- An **item tower** maps a movie (its ID **+ its genres**) -> an embedding.
- Train so a user's embedding is close to the embeddings of movies they liked.
- Recommend by nearest-neighbor: encode the user once, find the closest item vectors.

The key difference from MF: the **item tower uses features (genres)**, so it can generalize
and reason about content, not just memorize an ID. This is the design behind large-scale
production recommenders (YouTube, ads, etc.).

**Runtime: GPU ON.** This notebook rebuilds train/test if starting fresh.

## 0. (If starting fresh) rebuild train/test — else skip

In [ ]:
import pandas as pd, numpy as np, os
try:
    train.shape; test.shape; print("train/test already loaded")
except NameError:
    if not os.path.exists('ml-25m/ratings.csv'):
        !wget -q https://files.grouplens.org/datasets/movielens/ml-25m.zip && unzip -q -o ml-25m.zip
    ratings = pd.read_csv('ml-25m/ratings.csv')
    pos = ratings[ratings.rating >= 4.0].copy()
    uc = pos.userId.value_counts(); ic = pos.movieId.value_counts()
    pos = pos[pos.userId.isin(uc[uc>=5].index) & pos.movieId.isin(ic[ic>=5].index)]
    pos = pos.sort_values(['userId','timestamp']).reset_index(drop=True)
    pos['rk'] = pos.groupby('userId').cumcount()
    pos['sz'] = pos.groupby('userId')['userId'].transform('size')
    pos['is_test'] = pos['rk'] >= (pos['sz']*0.8)
    train = pos[~pos.is_test]; test = pos[pos.is_test]
    print(f"rebuilt -> train {len(train):,}  test {len(test):,}")

## 1. Load movie genres (the item features)
MovieLens ships a `movies.csv` with a pipe-separated genre list per movie
(e.g. "Action|Adventure|Sci-Fi"). We multi-hot encode genres: each movie becomes a
vector of 0/1 flags over the ~20 genre types. This is what the item tower will consume.

In [ ]:
movies = pd.read_csv('ml-25m/movies.csv')  # movieId, title, genres

# build genre vocabulary
all_genres = set()
for g in movies.genres:
    for tok in g.split('|'):
        all_genres.add(tok)
all_genres.discard('(no genres listed)')
genre_list = sorted(all_genres)
g2idx = {g:i for i,g in enumerate(genre_list)}
n_genres = len(genre_list)
print(f"{n_genres} genres:", genre_list)

# movieId -> multi-hot genre vector
def genre_vec(genres_str):
    v = np.zeros(n_genres, dtype=np.float32)
    for tok in genres_str.split('|'):
        if tok in g2idx: v[g2idx[tok]] = 1.0
    return v
movie_genre = {row.movieId: genre_vec(row.genres) for row in movies.itertuples()}

## 2. ID mappings + per-item genre feature matrix

In [ ]:
user_ids = train.userId.unique()
item_ids = train.movieId.unique()
u2idx = {u:i for i,u in enumerate(user_ids)}
i2idx = {m:i for i,m in enumerate(item_ids)}
n_users, n_items = len(u2idx), len(i2idx)

# genre feature matrix aligned to item index (0..n_items-1)
item_genres = np.zeros((n_items, n_genres), dtype=np.float32)
for m, idx in i2idx.items():
    if m in movie_genre: item_genres[idx] = movie_genre[m]

tr = train[train.movieId.isin(i2idx)]
train_u = tr.userId.map(u2idx).to_numpy()
train_i = tr.movieId.map(i2idx).to_numpy()
print(f"{n_users:,} users, {n_items:,} items, feature dim {n_genres}, pairs {len(train_u):,}")

## 3. The two towers
- **User tower**: an embedding table (one learned vector per user) -> MLP -> user embedding.
- **Item tower**: an item-ID embedding **concatenated with the movie's genre vector**,
  fed through an MLP -> item embedding.
Both output vectors of the same size `DIM`, so we can compare them by dot product.

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print("device:", device)

DIM = 64

class UserTower(nn.Module):
    def __init__(self, n_users, dim=DIM):
        super().__init__()
        self.emb = nn.Embedding(n_users, dim)
        self.net = nn.Sequential(nn.Linear(dim, dim), nn.ReLU(), nn.Linear(dim, dim))
        nn.init.normal_(self.emb.weight, std=0.05)
    def forward(self, u):
        return self.net(self.emb(u))

class ItemTower(nn.Module):
    def __init__(self, n_items, n_genres, dim=DIM):
        super().__init__()
        self.emb = nn.Embedding(n_items, dim)
        self.net = nn.Sequential(nn.Linear(dim + n_genres, dim), nn.ReLU(), nn.Linear(dim, dim))
        nn.init.normal_(self.emb.weight, std=0.05)
    def forward(self, i, genres):
        x = torch.cat([self.emb(i), genres], dim=1)
        return self.net(x)

user_tower = UserTower(n_users).to(device)
item_tower = ItemTower(n_items, n_genres).to(device)
item_genres_t = torch.tensor(item_genres, device=device)   # (n_items, n_genres)
print(user_tower); print(item_tower)

## 4. Train with in-batch softmax (sampled-softmax) loss
Two-tower models are usually trained with **in-batch negatives**: within a batch of
(user, liked-item) pairs, each user's *own* item is the positive, and the *other users'*
items in the same batch serve as negatives — for free. We maximize the score of the true
pair relative to the others via cross-entropy. This is efficient and is standard for
production two-tower retrieval.

In [ ]:
train_u_t = torch.tensor(train_u, dtype=torch.long)
train_i_t = torch.tensor(train_i, dtype=torch.long)
N = len(train_u_t)

params = list(user_tower.parameters()) + list(item_tower.parameters())
opt = torch.optim.Adam(params, lr=0.001, weight_decay=1e-6)
BATCH = 4096
EPOCHS = 12
TEMP = 0.07   # temperature: sharpens the softmax over in-batch items

for epoch in range(EPOCHS):
    perm = torch.randperm(N)
    total = 0.0
    user_tower.train(); item_tower.train()
    for start in range(0, N, BATCH):
        idx = perm[start:start+BATCH]
        u = train_u_t[idx].to(device)
        i = train_i_t[idx].to(device)

        ue = F.normalize(user_tower(u), dim=1)                 # (B, DIM)
        ie = F.normalize(item_tower(i, item_genres_t[i]), dim=1)  # (B, DIM)

        # similarity matrix: user b vs every item in the batch
        logits = (ue @ ie.t()) / TEMP                         # (B, B)
        labels = torch.arange(len(idx), device=device)        # correct item is on the diagonal
        loss = F.cross_entropy(logits, labels)

        opt.zero_grad(); loss.backward(); opt.step()
        total += loss.item() * len(idx)
    print(f"epoch {epoch+1}/{EPOCHS}  loss: {total/N:.4f}")

## 5. Recommend + evaluate (same metrics, same comparison)
Precompute every item embedding once. For a user, encode them, dot against all item
embeddings, mask training-seen items, take top-K. Compare to baseline + MF.

In [ ]:
user_tower.eval(); item_tower.eval()

from collections import defaultdict
seen = defaultdict(set)
for u,i in zip(train_u, train_i): seen[u].add(i)

@torch.no_grad()
def all_item_embeddings():
    embs = []
    for start in range(0, n_items, 8192):
        idx = torch.arange(start, min(start+8192, n_items), device=device)
        e = F.normalize(item_tower(idx, item_genres_t[idx]), dim=1)
        embs.append(e)
    return torch.cat(embs, 0)   # (n_items, DIM)

ITEM_EMB = all_item_embeddings()

@torch.no_grad()
def recommend_tt(user_id, k=10):
    if user_id not in u2idx: return []
    ui = u2idx[user_id]
    ue = F.normalize(user_tower(torch.tensor([ui], device=device)), dim=1)  # (1,DIM)
    scores = (ue @ ITEM_EMB.t()).squeeze(0)                                 # (n_items,)
    scores[list(seen[ui])] = -1e9
    top = torch.topk(scores, k).indices.cpu().numpy()
    return [item_ids[t] for t in top]

def dcg(hits): return sum(1.0/np.log2(i+2) for i,h in enumerate(hits) if h)
def evaluate(fn, k=10, n_users=3000, seed=0):
    tbu = test.groupby('userId').movieId.apply(set).to_dict()
    users = list(tbu.keys())
    sample = np.random.default_rng(seed).choice(users, size=min(n_users,len(users)), replace=False)
    R,Nd=[],[]
    for u in sample:
        truth=tbu[u]
        if not truth: continue
        recs=fn(u,k)
        if not recs: continue
        hits=[1 if m in truth else 0 for m in recs]
        R.append(sum(hits)/min(len(truth),k))
        ideal=dcg([1]*min(len(truth),k)); Nd.append(dcg(hits)/ideal if ideal>0 else 0.0)
    return float(np.mean(R)), float(np.mean(Nd))

rec, ndcg_score = evaluate(recommend_tt)
print(f"TWO-TOWER          ->  Recall@10: {rec:.4f}   NDCG@10: {ndcg_score:.4f}")
print("compare:  popularity 0.0700/0.0632   |   MF 0.0825/0.0743")

## 6. Read the result
If two-tower beats MF, the genre-aware item tower is adding real signal. If it's about
the same, that's an honest and common finding on MovieLens (IDs alone already capture a
lot; genres help most for cold-start / sparse items). Either way we report it straight
and can explain *why*.

**Next (Stage 4):** a re-ranking layer — use two-tower for fast candidate retrieval, then
a ranker to reorder the shortlist. Then Stage 5: serve it as a FastAPI + Next.js app.